# Organize Imports and Dependencies

In [1]:
import pandas as pd

# Data Preprocessing and Cleaning

In [3]:
# Awards Data
dfa = pd.read_csv('df_all_award_results.csv')

# Transactions Data
df_transactions = pd.read_csv('df_transactions.csv')

In [7]:
# Calculate aggregate statistics at parent (Award ID) level
df_t_gb = df_transactions.groupby('generated_internal_id').agg({'Transaction Amount' : 'sum',
                                                     'Award ID' : 'count'}).reset_index()

In [ ]:
# Merge to df_awards
df_merge = pd.merge(dfa, df_t_gb, how = 'left', on = 'generated_internal_id')

In [13]:
# Create Democrat / Republican column
def is_democrat(date_input):
    # Biden term: Jan 20, 2021 to Jan 20, 2025
    biden_start = pd.Timestamp('2021-01-20')
    biden_end = pd.Timestamp('2025-01-20')
    
    # Check if the row's date falls within the Biden administration
    if biden_start <= pd.Timestamp(date_input) < biden_end:
        return 1
    else:
        return 0
    
df_merge['is_democrat'] = df_merge['Start Date'].apply(lambda row : is_democrat(row))

In [16]:
# Filter for modifications to the contract (ie not base amount where tx = 1)
df_transactions[df_transactions['Mod'] != '0']

,Unnamed: 0,Award ID,Recipient Name,Action Date,Transaction Amount,Awarding Agency,PSC,generated_internal_id,internal_id
0,0,70FB8021F00000016,CSRA LLC,2021-02-15,1.443840e+08,Department of Homeland Security,"{'code': 'R702', 'description': 'SUPPORT- MANA...",CONT_AWD_70FB8021F00000016_7022_GS35F393CA_4732,291212909
1,1,70T03018F2BCIO660,CACI INC - FEDERAL,2020-11-12,5.631591e+07,Department of Homeland Security,"{'code': 'D316', 'description': 'IT AND TELECO...",CONT_AWD_70T03018F2BCIO660_7013_HSHQDC13DE2095...,291253091
2,2,70B04C20F00001348,"LEIDOS, INC.",2021-03-31,4.905354e+07,Department of Homeland Security,"{'code': 'D308', 'description': 'IT AND TELECO...",CONT_AWD_70B04C20F00001348_7014_70B04C20A00000...,291183250
3,3,70B04C20F00000111,FOUR LLC,2021-03-30,4.865881e+07,Department of Homeland Security,"{'code': '7030', 'description': 'INFORMATION T...",CONT_AWD_70B04C20F00000111_7014_NNG15SC73B_8000,291182960
4,4,70SBUR19F00000062,"SEV1TECH, LLC",2021-02-03,4.689422e+07,Department of Homeland Security,"{'code': 'D309', 'description': 'IT AND TELECO...",CONT_AWD_70SBUR19F00000062_7003_GS35F0027M_4730,291248817
...,...,...,...,...,...,...,...,...,...
665934,665934,9523ZY25F0023,SOFTWARE INFORMATION RESOURCE CORP.,2026-04-16,1.037556e+04,Commodity Futures Trading Commission,"{'code': 'DA10', 'description': 'IT AND TELECO...",CONT_AWD_9523ZY25F0023_9507_NNG15SD74B_8000,292210546
665935,665935,9523ZY25F0018,SHIVOY INC.,2026-04-17,5.222000e+03,Commodity Futures Trading Commission,"{'code': 'R425', 'description': 'SUPPORT- PROF...",CONT_AWD_9523ZY25F0018_9507_47QTCB21D0061_4732,292210541
665936,665936,9523ZY26F0022,DYNAMIC SYSTEMS INC,2026-04-13,0.000000e+00,Commodity Futures Trading Commission,"{'code': '7A21', 'description': 'IT AND TELECO...",CONT_AWD_9523ZY26F0022_9507_NNG15SC69B_8000,356948944
665937,665937,9523ZY25F0006,NR LABS LLC,2026-04-09,0.000000e+00,Commodity Futures Trading Commission,"{'code': 'R499', 'description': 'SUPPORT- PROF...",CONT_AWD_9523ZY25F0006_9507_9523ZY25A0001_9507,292210531


# Feature Engineering

In [ ]:
# Create a binary column (overspend) if a contract will cost more than the original base award (total award amount > base award amount).